# 🌐 Run Virgo-1.0-Angkor for the Virgo AI website — Kaggle (free test server)

Kaggle gives a free GPU for about **30 hours a week** (separate from Colab's limit). This runs your trained **Virgo-1.0-Angkor** here and gives it a public https address, so virgoai.camksn.com can use it for **chat** and **real-time voice (Virgo Live)**.

**First time only**
1. Kaggle → your profile → **Settings → Phone verification** (needed for GPU and internet).
2. In this notebook: **Add-ons → Secrets → Add secret**: label `HF_TOKEN`, value your Hugging Face token (`hf_...`), and tick it for this notebook.

**Each run**
1. Right panel → **Session options**: **Accelerator → GPU T4 x2** and **Internet → On**.
2. **Run all**.
3. Copy the two values it prints into Cloudflare → your Worker → **Settings → Variables and Secrets** as **Secrets**: `VIRGO_API_URL` and `VIRGO_API_KEY`.

⚠️ Test only: it works while this session runs (up to 12 hours, keep the tab open), and the address changes each run. The last cell keeps the session busy so Kaggle doesn't stop it as idle. Stop the session when you're done to save your GPU hours.

**Your own fixed address (optional):** create a Cloudflare Tunnel (Zero Trust → Networks → Tunnels → Create → Cloudflared, public hostname `virgo.camksn.com` → HTTP `localhost:8000`), then add two secrets here (**Add-ons → Secrets**): `CF_TUNNEL_TOKEN` (the tunnel token) and `VIRGO_API_KEY` (a long password you choose). Set the Worker secrets once to `https://virgo.camksn.com` and the same password; after that, just run this notebook.

In [ ]:
# ⚙️ Settings
HF_MODEL = ""  # leave empty to use <your Hugging Face name>/Virgo-1.0-Angkor
VIRGO_HOST = "https://virgo.camksn.com"  # your own address (named Cloudflare Tunnel), used when the CF_TUNNEL_TOKEN secret is set

In [ ]:
# 1️⃣ Set up
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "❌ No GPU: Session options → Accelerator → GPU T4 x2"
subprocess.run("pkill -f 'uvicorn serve.app' ; pkill -f cloudflared", shell=True)
WORK = "/kaggle/working"
os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    r = subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"])
    assert r.returncode == 0, "❌ Can't download: Session options → Internet → On (needs phone verification)"
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-serve.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kokoro"])  # natural English voice; optional (MMS is used if it fails)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # conflicts with newer peft
from huggingface_hub import login, HfApi, snapshot_download
from getpass import getpass
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None
login(token=token or getpass("Hugging Face token (hf_...): ").strip())
ADAPTER = "chat/out/virgo-1.0-chat-lora"
if not os.path.isdir(ADAPTER):
    me = HfApi().whoami()['name']
    # Virgo-1.0-Angkor-12B (chat/train_kaggle.ipynb) when you've trained it, else the 4B.
    repo = HF_MODEL or next((r for r in (f"{me}/Virgo-1.0-Angkor-12B", f"{me}/Virgo-1.0-Angkor")
                             if HfApi().file_exists(r, "adapter_config.json")), f"{me}/Virgo-1.0-Angkor")
    print("⬇️ Downloading", repo)
    snapshot_download(repo, local_dir=ADAPTER)
# Virgo's own Khmer hearing (speech/train_hearing_kaggle.ipynb), when you've trained it.
HEARING = "speech/out/virgo-1.0-stt"
if not os.path.isdir(HEARING):
    try:
        hearing_repo = f"{HfApi().whoami()['name']}/Virgo-1.0-Angkor-Hearing"
        if HfApi().file_exists(hearing_repo, "config.json"):
            print("⬇️ Downloading Virgo's Khmer hearing")
            snapshot_download(hearing_repo, local_dir=HEARING)
    except Exception as err:
        print("Virgo's own hearing not used:", err)
print("✅ Ready: Virgo-1.0-Angkor is in", ADAPTER, "| hearing:", "Virgo's own" if os.path.isdir(HEARING) else "Whisper large-v3-turbo")

In [ ]:
# 🗣️ Virgo-1.0-Angkor-Voice: Virgo's voice on VoxCPM2 (Apache 2.0, clear Khmer + English)
# Uses your designed voice (speech/design_voice.ipynb → you/Virgo-1.0-Angkor-Voice) when it exists,
# otherwise a described warm, clear voice. Set USE_VIRGO_VOICE = False to use the older voices.
USE_VIRGO_VOICE = True
import os, subprocess, sys, torch
from huggingface_hub import HfApi
if USE_VIRGO_VOICE:
    env_dir = "/kaggle/working/vox-env"
    python = f"{env_dir}/bin/python"
    if not os.path.exists(python):
        print("⏳ Installing VoxCPM2 for Virgo's voice (3–5 minutes, first run only)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run(["uv", "venv", "-q", "--python", "3.11", env_dir], check=True)
        r = subprocess.run(["uv", "pip", "install", "-q", "--python", python, "git+https://github.com/OpenBMB/VoxCPM", "soundfile", "huggingface_hub"], capture_output=True, text=True)
        if r.returncode:  # GitHub unreachable: the pip release
            r = subprocess.run(["uv", "pip", "install", "-q", "--python", python, "voxcpm", "soundfile", "huggingface_hub"], capture_output=True, text=True)
        if r.returncode:
            print(r.stderr[-3000:]); raise SystemExit("❌ VoxCPM2 didn't install: send the lines above to Claude.")
    # Download VoxCPM2 now (about 5 GB) so the first answer isn't slow.
    subprocess.run([python, "-c", "from huggingface_hub import snapshot_download; snapshot_download('openbmb/VoxCPM2')"], check=True)
    os.environ["VIRGO_VOX_PYTHON"] = python
    os.environ["VIRGO_VOX_REPO"] = f"{HfApi().whoami()['name']}/Virgo-1.0-Angkor-Voice"
    if torch.cuda.device_count() > 1:
        os.environ["VIRGO_VOX_DEVICE"] = "cuda:1"  # the second T4: chat and hearing keep the first
    try:
        designed = HfApi().file_exists(os.environ["VIRGO_VOX_REPO"], "voice.wav")
    except Exception:
        designed = False
    print("✅ Virgo speaks with", "your designed voice from " + os.environ["VIRGO_VOX_REPO"] if designed else "the described voice (design yours with speech/design_voice.ipynb)",
          "| GPU:", os.environ.get("VIRGO_VOX_DEVICE", "cuda:0"))
    if torch.cuda.device_count() == 1:
        # One GPU and ~12 GB of RAM (Colab): chat in 4 bits and no torch.compile for the voice, so all fit.
        os.environ["VIRGO_CHAT_4BIT"] = "1"
        os.environ["VIRGO_VOX_OPTIMIZE"] = "0"
        print("ℹ️ One GPU: chat runs in 4 bits so Virgo's voice fits too. Use the 4B Virgo-1.0-Angkor (HF_MODEL in Settings).")


In [ ]:
# 🎙️ Virgo's own Khmer voice (when trained with speech/train_voice_kaggle.ipynb)
import os, subprocess, sys
from huggingface_hub import HfApi
VOICE_REPO = f"{HfApi().whoami()['name']}/Virgo-1.0-Angkor-Voice"
try:
    has_voice = HfApi().file_exists(VOICE_REPO, "model.pth")
except Exception:
    has_voice = False
if has_voice:
    # The voice needs its own Python environment (older libraries than the Virgo server).
    env_dir = "/kaggle/working/voice-env"
    python = f"{env_dir}/bin/python"
    if not os.path.exists(python):
        print("⏳ Installing Virgo's Khmer voice (2–4 minutes, first run only)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run(["uv", "venv", "-q", "--python", sys.executable, env_dir], check=True)
        subprocess.run(["uv", "pip", "install", "-q", "--python", python, "--index-url", "https://download.pytorch.org/whl/cpu", "torch==2.8.*", "torchaudio==2.8.*"], check=True)
        subprocess.run(["uv", "pip", "install", "-q", "--python", python, "coqui-tts==0.27.2", "soundfile"], check=True)
    os.environ["VIRGO_VOICE_PYTHON"] = python
    os.environ["VIRGO_VOICE_REPO"] = VOICE_REPO
    print("✅ Khmer answers will use Virgo's own voice from", VOICE_REPO)
else:
    print("ℹ️ No trained Virgo voice yet: Khmer uses the MMS voice (non-commercial). Train one with speech/train_voice_kaggle.ipynb.")

In [ ]:
# 2️⃣ Start the Virgo server and open a public address
import os, subprocess, time, re, secrets, json, urllib.request
WORK = "/kaggle/working"
os.chdir(f"{WORK}/virgo-models")
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
SAVED_KEY = secret("VIRGO_API_KEY")
KEY = SAVED_KEY or secrets.token_urlsafe(32)  # a saved key stays the same every run
TUNNEL = secret("CF_TUNNEL_TOKEN")           # your own tunnel: the address never changes
# One GPU is enough; on "T4 x2" this keeps Virgo on the first one.
env = {**os.environ, "VIRGO_API_KEY": KEY, "VIRGO_PRELOAD": "chat,speech", "CUDA_VISIBLE_DEVICES": "0"}
subprocess.Popen(f"uvicorn serve.app:app --host 0.0.0.0 --port 8000 > {WORK}/server.log 2>&1", shell=True, env=env)
print("⏳ Loading Virgo-1.0-Angkor (1–3 minutes)...")
for _ in range(180):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5); break
    except Exception:
        pass
else:
    print("".join(open(f"{WORK}/server.log").readlines()[-40:])); raise SystemExit("❌ The server didn't start: send the lines above to Claude.")
req = urllib.request.Request("http://localhost:8000/v1/chat", method="POST",
    data=json.dumps({"messages": [{"role": "user", "content": "Who are you?"}], "max_tokens": 60}).encode(),
    headers={"content-type": "application/json", "authorization": f"Bearer {KEY}"})
try:
    print("🤖", json.loads(urllib.request.urlopen(req, timeout=600).read())["reply"])
    # Which voice Virgo speaks with (and why, if it isn't Virgo-1.0-Angkor-Voice)
    for line in open("../server.log", errors="replace"):
        if any(k in line for k in ("🗣️ Voice", "Virgo-1.0-Angkor-Voice", "Virgo voice", "VoxCPM", "Running on device")):
            print(line.rstrip()[:300])
except Exception as err:
    print("".join(open(f"{WORK}/server.log").readlines()[-40:])); raise SystemExit(f"❌ {err}: send the lines above to Claude.")
CF = "/kaggle/cloudflared"
if not os.path.exists(CF):
    subprocess.run(f"wget -q -O {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {CF}", shell=True)
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)
log = "/kaggle/working/tunnel.log"
if TUNNEL:
    # The token goes in the environment, never on the command line or in the output.
    subprocess.Popen(f"{CF} tunnel --no-autoupdate run > {log} 2>&1", shell=True, env={**os.environ, "TUNNEL_TOKEN": TUNNEL})
    for _ in range(60):
        time.sleep(2)
        if "Registered tunnel connection" in open(log).read(): break
    else:
        print(open(log).read()[-2000:]); raise SystemExit("❌ Your tunnel didn't connect: check the CF_TUNNEL_TOKEN secret (lines above).")
    url = VIRGO_HOST.rstrip("/")
else:
    subprocess.Popen(f"{CF} tunnel --url http://localhost:8000 > {log} 2>&1", shell=True)
    url = None
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log).read())
        if m: url = m.group(0); break
    if not url:
        print(open(log).read()[-2000:]); raise SystemExit("❌ No public address: send the lines above to Claude.")
print("\n✅ Virgo server is live (chat + real-time voice) at", url)
if TUNNEL and SAVED_KEY:
    print("Your own address and saved key: nothing to change in Cloudflare (VIRGO_API_URL =", url + ", VIRGO_API_KEY = your saved key).")
else:
    print("Add these as Cloudflare Worker SECRETS:")
    print("VIRGO_API_URL =", url)
    print("VIRGO_API_KEY =", "your saved VIRGO_API_KEY secret" if SAVED_KEY else KEY)


In [ ]:
# 3️⃣ Keep the session awake (stop this cell, or the session, when you're done)
import time, urllib.request
while True:
    time.sleep(300)
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=10)
        print(time.strftime("%H:%M"), "✅ Virgo server running")
    except Exception as err:
        print(time.strftime("%H:%M"), "⚠️ server not answering:", err, "— see /kaggle/working/server.log")